# UBCF

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import train_test_split

In [ ]:
uid = 25
top_k = 10

# Load data
ratings = pd.read_csv("mlsmall/ratings.csv")
movies = pd.read_csv("mlsmall/movies.csv")

train, test = train_test_split(ratings, test_size=0.2, random_state=42)

# User-Movie matrix
R = train.pivot(index="userId", columns="movieId", values="rating")

# Mean-center
mean = R.mean(axis=1)
Rc = R.sub(mean, axis=0).fillna(0)

# User similarity
sim = cosine_similarity(Rc)
sim = pd.DataFrame(sim, index=R.index, columns=R.index)

# Prediction function
def predict(uid, movie):
    if uid not in R.index or movie not in R.columns:
        return mean.mean()

    neighbors = sim.loc[uid].drop(uid).nlargest(10)

    num = den = 0
    for u, s in neighbors.items():
        if pd.notna(R.loc[u, movie]):
            num += s * (R.loc[u, movie] - mean[u])
            den += abs(s)

    return mean[uid] + num / den if den else mean[uid]

# Predict train and test ratings
train_pred = [predict(u, m) for u, m in zip(train.userId, train.movieId)]
test_pred  = [predict(u, m) for u, m in zip(test.userId, test.movieId)]

# Evaluation
print("Train MAE :", mean_absolute_error(train.rating, train_pred))
print("Train RMSE:", np.sqrt(mean_squared_error(train.rating, train_pred)))

print("Test MAE  :", mean_absolute_error(test.rating, test_pred))
print("Test RMSE :", np.sqrt(mean_squared_error(test.rating, test_pred)))

# Top-10 recommendations for user
pred = {}

for movie in R.columns[R.loc[uid].isna()]: pred[movie] = predict(uid, movie)

top10 = sorted(pred.items(), key=lambda x: x[1], reverse=True)[:top_k]

print("\nTop K Recommendations:")
for movie, rating in top10:
    title = movies.loc[movies.movieId == movie, "title"].iloc[0]
    print(title, ":", round(rating, 2))

Train MAE : 0.6653184661151293
Train RMSE: 0.8634798499370169
Test MAE  : 0.7440345950144934
Test RMSE : 0.9744301407437678

Top K Recommendations:
Three Billboards Outside Ebbing, Missouri (2017) : 6.92
Sense and Sensibility (1995) : 6.56
Babe (1995) : 6.56
What's Eating Gilbert Grape (1993) : 6.56
Godfather: Part II, The (1974) : 6.32
Godfather, The (1972) : 6.11
Star Wars: Episode VI - Return of the Jedi (1983) : 6.08
Full Metal Jacket (1987) : 6.08
L.A. Confidential (1997) : 6.08
Game, The (1997) : 6.08
